# Notes — Chapter 3: Data Ethics (fastbook)

**Core idea:** a model can be buggy, meet data it never saw, or work exactly as designed and still be used for harm. Whoever trains the model is best placed to explain its limits, so ethics is part of the practitioner's job, not someone else's.

Ethics gives no list of right answers. It's context-dependent, needs many stakeholders' perspectives, and is best practiced as a team.

## 1. Three anchor examples

| Issue | Case | Lesson |
|---|---|---|
| **Recourse** | Arkansas Medicaid algorithm cut care hours (e.g. a patient with cerebral palsy lost ~20 hrs/week); a court later found implementation bugs | People affected by automated decisions need explanations and a way to appeal |
| **Feedback loops** | YouTube's watch-time-optimizing recommender (~70% of views) amplified conspiracy content | When a model shapes its own future data, small effects compound |
| **Bias** | Latanya Sweeney: searching historically Black names surfaced "arrested?" ads; white names got neutral ads | Systems can encode and spread social bias with real consequences (e.g. hiring) |

## 2. Why should a data scientist care?

- **IBM & Nazi Germany:** IBM supplied, configured, and serviced punch-card tabulation used to track people through the camps. Ordinary employees "just doing their jobs" were part of it.
- **Volkswagen emissions scandal:** the first person jailed was an engineer (James Liang), not an executive. Following orders is not a shield.
- **Optimizing a metric ≠ good outcomes.** Consider the whole pipeline from research → product → decision.
- "If I don't do it, someone else will" — if you were chosen, you're the best candidate; refusing still matters. Saying **no** is sometimes the right answer.

## 3. ML is product design

Choices like aggregation level, loss function, validation set, speed vs. accuracy, and out-of-domain handling are **product decisions**, but the downstream users (managers, judges, doctors, police) can't see or change them.

- **Amazon Rekognition:** studies found inaccurate and racially biased results; it falsely matched 28 members of Congress to mugshots (disproportionately people of color). Amazon blamed default settings it hadn't told police to change.
- **Takeaway:** data scientists belong on cross-disciplinary teams, close to the people who use their work. Large companies' compartmentalization makes this harder.

## 4. Topics in data ethics

### 4.1 Recourse & accountability
- Complex systems diffuse responsibility; in Arkansas, the algorithm's creator, officials, and implementers all blamed each other. danah boyd: algorithmic systems extend bureaucracy's ability to evade responsibility.
- Data contains errors, so **audits and correction mechanisms** are essential.
  - CalGang database: 42 infants listed, with no process to remove anyone.
  - FTC 2012: 26% of consumers had at least one credit-report error; 5% had serious ones. Fixing them is slow and opaque.

### 4.2 Feedback loops
A model's predictions change the world, which changes the next round of training data, which pushes predictions further the same way.

- **YouTube + conspiracies:** controversial content holds attention → recommended more → heavy viewers drawn in → more of it recommended.
- **YouTube + children's videos (NYT 2019):** the recommender grouped innocent home videos of children for predatory viewers. Nobody designed this; it emerged from **metric optimization**, and people actively exploit such edge cases.
- **RT and the Mueller report:** an outlier in how many channels recommended it, suggesting the algorithm was gamed. Opacity makes this hard to detect.
- **Loops without humans (Aurélien Géron):** video topic ← channel topic ← video topics. A misclassified "photography" video can snowball. *Fix:* classify videos without the channel signal, and use only those labels to classify channels.
- **Good example — Meetup:** men showed more interest in tech meetups; using gender would show women fewer → fewer attend → even fewer shown. They chose **not to use gender**. The most optimal model isn't necessarily the one to ship.
- **Bad example — Facebook groups:** joining one conspiracy group leads to recommendations for others.

➡ Anticipate loops; act at the first sign of one.

In [ ]:
# Toy simulation: how a small initial skew snowballs in a feedback loop.
# Two content types start nearly equal; the recommender shows content in
# proportion to past engagement, and type B is slightly more "engaging".
import numpy as np

share_B = 0.50          # initial share of recommendations going to B
engage = {'A': 1.00, 'B': 1.10}   # B gets 10% more engagement per view

history = [share_B]
for step in range(30):
    clicks_A = (1 - share_B) * engage['A']
    clicks_B = share_B * engage['B']
    share_B = clicks_B / (clicks_A + clicks_B)   # retrain on engagement
    history.append(share_B)

for t in [0, 5, 10, 20, 30]:
    print(f"step {t:2d}: share of B = {history[t]:.2f}")

*A 10% engagement edge turns into near-total dominance — no one "decided" that B should win.*

### 4.3 Bias
"Bias" here is the **social-science** sense, not statistical bias or a model's bias parameters. Suresh & Guttag (MIT) list six sources:
**historical, representation, measurement, aggregation, evaluation, deployment** bias. The chapter covers four:

| Type | What it is | Example |
|---|---|---|
| **Historical** | Society and its processes were already biased; exists even with perfect sampling | COMPAS recidivism tool (ProPublica); Google Photos labeling Black users as "gorillas" (Google's fix was just removing the label); Google Translate Turkish "o" → he/she by job stereotype; Facebook showing housing ads to different races |
| **Representation** | Model *amplifies* an imbalance rather than just reflecting it | Bias in Bios: women were 14.6% of surgeons in training data but only 11.6% of the model's true positives |
| **Measurement** | Measuring the wrong thing, or in the wrong way | Stroke predictors from EHR included accidental injury, colonoscopy, sinusitis — it measured *who visits doctors*, not *who has strokes* |
| **Aggregation** | Model omits key factors / interactions | Diabetes: complications and HbA1c differ across ethnicities and genders, but treatment models often ignore this |

Related findings:
- **Gender Shades (Buolamwini):** IBM's error rate was 34.7% for darker-skinned women vs 0.3% for lighter-skinned men. After publicity, vendors largely closed the gap within a year, proving it was a **data/testing problem**, not something inherent.
- **Geodiversity:** ImageNet / Open Images are mostly US & Western Europe. Models misread household items (e.g. soap) from low-income countries.
- Teams are mostly young white men who user-test on friends and family, so these blind spots are unsurprising.

### 4.4 Addressing bias
- Different bias types need different fixes: more diverse data helps **representation** bias but not **historical** or **measurement** bias.
- No dataset is bias-free → **document datasets** (how/why created, appropriate uses, limitations), e.g. *Datasheets for Datasets*.

**"Humans are biased too, so why care?"** Algorithms differ from people:
- They create **feedback loops** and **amplify** bias.
- They're used differently: assumed objective, often deployed **without appeals**, applied **at scale**, and **cheap**.
- Cathy O'Neil: the privileged get processed by people, the poor by algorithms.
- Technology is power, which brings responsibility.

### 4.5 Disinformation
- Not just "fake news": the aim is often to sow confusion and make people give up on finding the truth. It typically mixes facts, half-truths, exaggerations, and lies (Ladislav Bittman).
- **2016 US election:** Russian trolls ("Heart of Texas") organized protests *and* counter-protests at the same place and time, without anyone showing up themselves.
- Coordinated inauthentic accounts fake consensus; people are strongly shaped by their perceived in-group.
- Deep learning makes generated text and forgery cheap. Proposed fix (Oren Etzioni): make **digital signatures** the norm for authenticating content.

## 5. Identifying & addressing ethical issues

Four steps: **analyze your project → set up processes → support good policy → increase diversity.**

### Rachel Thomas's project checklist
1. Should we even be doing this?
2. What bias is in the data?
3. Can the code and data be audited?
4. What are error rates for different subgroups?
5. How accurate is a simple rule-based alternative?
6. What processes handle appeals or mistakes?
7. How diverse is the team that built it?

Also: **data gets repurposed.** IBM helped with the 1933 German census; US census data was used to intern Japanese-Americans. Assume stored personal data may be taken by governments or thieves (Tim Wu).

### Processes (Markkula Center toolkit)
- Regular **ethical risk sweeps** (like penetration testing)
- **Expand the ethical circle**: which stakeholders' interests have we assumed instead of asking about? Who's indirectly affected? Who might use this in unintended ways?
- **Think about the terrible people**: how could bad actors abuse, steal, misread, hack, or weaponize this?

### The five ethical lenses
| Lens | Key question |
|---|---|
| Rights | Which option best respects everyone's rights? |
| Justice | Which treats people equally or proportionately? |
| Utilitarian | Which does the most good and least harm? |
| Common good | Which serves the whole community? |
| Virtue | Which makes me the person I want to be? |

Two broader perspectives: **consequentialist** (who's affected, net good vs harm, future generations, does harm fall on the least powerful, dual use?) and **deontological** (rights, duties, dignity, autonomy, trust, conflicting obligations).

### Diversity
- Fewer than 12% of AI researchers are women. Homogeneous teams share blind spots.
- **Tracy Chou (Quora)** pushed for a block button early because she'd experienced harassment herself.
- Women leave tech at about twice the rate of men (41% vs 17%) due to unfair treatment: vaguer feedback, fewer stretch assignments, and mentorship that tells them to change rather than endorsing them.
- Teaching girls to code won't fix this while women keep leaving. All 60 women of color interviewed in STEM research reported discrimination.
- **Hiring:** Triplebyte found companies hire people who resemble their founders, not people who fit their actual needs. DL groups founded by academics favor math and jargon over real problem-solving, which creates an opening for companies that value results over pedigree.

### FAccT (Fairness, Accountability, Transparency)
- ACM conference; Microsoft FATE group; reference book: *Fairness and Machine Learning* (Barocas, Hardt, Narayanan).
- Limitation: a narrow technical framing can sidestep questions of **power**.
- **"A Mulching Proposal"** (Keyes et al.) satirically "improves" a system that turns elderly people into slurry until it scores well on FAT, showing a framework can pass something monstrous.
- **Philosophy tool:** test any framework by finding a case where it gives an answer no one could accept, then refine it.

## 6. Role of policy

Technical and design fixes help but aren't enough while harmful tech stays profitable. **Incentives must change.**

- **Facebook in Myanmar:** warned from 2013 that it was enabling hate speech; by end of 2015 it had only **4** Burmese-speaking contractors; the UN found it played a "determining role" in the Rohingya genocide.
- **Facebook in Germany:** quickly hired **1,200** moderators to avoid fines of up to €50M under a hate-speech law.
→ Financial penalties moved the company faster than genocide did.

- **Maciej Cegłowski:** privacy needs the same kind of coordinated regulation that fixed smog, burning rivers, leaded gasoline, and ozone depletion. Individual consumer choices couldn't fix those.
- Privacy and a healthy information ecosystem are **public goods**. Many tech harms are really **human rights issues** (biased sentencing, age-targeted job ads, facial recognition of protesters), and the law is the right venue for them.
- We need **both** regulation **and** ethical individuals, because the law can't cover every edge case.

### Precedent: car safety
Seatbelts, collapsible steering columns, and crash-test dummies took decades of activism; GM even hired private detectives to dig up dirt on Ralph Nader. Female crash-test dummies were only required from **2011**; before that, women were **40% more likely** to be injured in an equivalent crash.

## 7. Conclusion
- Ethics lacks binary answers, but there are clear pitfalls to avoid and good practices to follow.
- Julia Angwin: we're still in the **diagnosis phase**, like the early industrial era before labor laws. Feeling your understanding is incomplete is normal.
- Fred Monroe (ex-hedge fund): distribution shift and feedback loops are also key to **profitable** trading models. Careful ethical thinking overlaps with building successful data products.

## 8. Questionnaire — my answers

1. **Does ethics provide a list of right answers?** No. It's context-dependent and involves many stakeholders; it's a skill to practice.
2. **Why work with people of different backgrounds?** They spot issues you'd miss and bring a wider range of solutions.
3. **IBM in Nazi Germany?** Supplied and serviced tabulation machines that tracked people through the camps (including codes for victim groups and causes of death). The company chased profit and acclaim from the top down; workers followed orders and did their jobs.
4. **First person jailed in the VW scandal?** An engineer, James Liang, who did what he was told.
5. **CalGang database problem?** Full of errors (42 infants, 28 marked as "admitting" gang membership) with no process to fix mistakes or remove people.
6. **Why did YouTube recommend children's videos to pedophiles?** The recommender optimized watch time and learned co-viewing patterns, so it clustered such videos for those viewers. It was an emergent result of the metric and exploited by users, not something anyone programmed.
7. **Problems with centrality of metrics?** Algorithms optimize the number at all costs, producing edge cases and feedback loops that people find and exploit, even when the number no longer reflects real goals.
8. **Why did Meetup drop gender?** To avoid a loop where women see fewer tech meetups → attend fewer → get shown even fewer.
9. **Six bias types (Suresh & Guttag)?** Historical, representation, measurement, aggregation, evaluation, deployment.
10. **Two examples of historical race bias in the US?** Doctors given identical files recommended cardiac catheterization less often for Black patients; Black car buyers got higher opening prices (~$700) and smaller concessions. (Also: fewer replies to rental inquiries under Black names; all-white juries convicting Black defendants more often.)
11. **Where are most ImageNet images from?** The US and Western Europe.
12. **Why does sinusitis predict stroke?** Measurement bias: the label is a *diagnosed* stroke, so the model learns who tends to visit doctors (and has access to care), not who actually has strokes.
13. **Representation bias?** When a model amplifies an existing imbalance (e.g. under-predicting women surgeons beyond the base rate), often because simple models assume an obvious correlation always holds.
14. **How are machines and people used differently for decisions?** Algorithms are assumed objective, often have no appeals process, run at scale, and are cheap. They can also create feedback loops and amplify bias.
15. **Is disinformation the same as fake news?** No. It often mixes truth and half-truths and aims to sow confusion and distrust, not just plant false beliefs.
16. **Why is auto-generated-text disinformation significant?** Deep learning makes convincing content cheap and scalable, enabling floods of fake accounts and consensus.
17. **Markkula's five lenses?** Rights, justice, utilitarian, common good, virtue.
18. **Where is policy appropriate?** Where profit incentives, externalities, public goods (privacy, information ecosystem), or human-rights issues are involved, i.e. problems individual or company action can't fix.

## 9. Further research — to do
- [ ] Read *What Happens When an Algorithm Cuts Your Healthcare* (The Verge); how could it be avoided?
- [ ] YouTube recommender: must recommender systems produce harmful loops? What could Google or governments do?
- [ ] Sweeney, *Discrimination in Online Ad Delivery*: is Google responsible?
- [ ] How can cross-disciplinary teams prevent harm?
- [ ] Mullainathan & Obermeyer, *Does ML Automate Moral Hazard and Error?*: what actions should follow?
- [ ] Etzioni, *How Will We Prevent AI-Based Forgery?*: would digital signatures work?
- [ ] Apply the 7-question checklist (§5) to my own project
- [ ] Could my team be more diverse? How?

## End of Part 1 — practice checklist
- [ ] Run notebook 1 on a GPU server (Kaggle) and try my own images
- [ ] Run notebook 2 with my own image-search dataset
- [ ] Brainstorm a DL use for my own projects, including data sources, likely problems, and mitigations